# The Bionic Bat Architecture

## 1. The SNN Transmitter (The "Source")

This is the biological heart of the transmission. Instead of a computer clock, we use a neuron.

Input: A constant current (DC drive).

Mechanism: An snntorch.Leaky neuron integrates this current. When it hits a threshold, it fires.

Output: A semi-regular spike train (sent_spikes). This creates a "Natural PRI" (Pulse Repetition Interval) that varies slightly based on neural dynamics, unlike a rigid machine clock.

## 2. The RF Front-End (Signal Generation)

We need to turn those spikes into a wave that can travel through air.

Convolution (Sifting): We "paste" a mathematical chirp (FMCW waveform) onto every single spike.

Modulation: We mix this baseband signal with a high-frequency carrier (e.g., 40kHz or 20GHz) to transmit it.

## 3. The Physical Channel (The "World")

This is the physics simulation.

Time-of-Flight: The signal travels to the object and back. Distance= 
2
Speed×Time
​	
 .

Interaural Delay (2D): If the target is to the right, the echo hits the Right Ear before the Left Ear.

Doppler Shift: If the target is moving, the frequency of the echo is stretched or squashed.

Noise & Attenuation: The signal gets weaker (inverse square law) and corrupted by static.

## 4. The Receiver & Encoder (The "Ears")

We capture the echo and turn it back into the language of the brain (spikes).

Matched Filter: A mathematical filter that looks for the specific "chirp" shape we sent. It compresses the energy back into a sharp peak.

SNN Encoder: A second snntorch.Leaky neuron "watches" this analog signal. When it sees a sharp peak from the Matched Filter, it fires. This creates recovered_spikes.

## 5. The SNN Classifier (The "Brain")

This is the trainable neural network (Notebook 5).

Input: It receives two parallel streams:

sent_spikes (The reference copy).

recovered_spikes (The delayed echo).

Hidden Layer: Neurons connect these two streams, learning to fire only when specific delay patterns occur (Coincidence Detection).

Output: A "Vote." The neuron corresponding to the correct distance (e.g., "Class 3: 3-4 meters") fires the most.

Diagrammatic Flow

### How to read this flow:

Pink Boxes are Spiking Neurons (snntorch).

The Dotted Line represents the "Efferenct Copy" — the brain keeping a copy of the command it just sent so it can compare it to the echo later.

The Flow is entirely Feed-Forward in our current model, meaning data flows strictly from the transmitter to the decision.

```mermaid
graph TD
    subgraph TRANSMITTER ["Phase 1: The Neural Source"]
    A[Constant Current] -->|Drive| B(LIF Neuron)
    B -->|Spikes| C[Convolution w/ Chirp]
    C -->|Modulation| D[Tx Signal out]
    end

    subgraph CHANNEL ["Phase 2: The Physics"]
    D --> E{Target?}
    E -->|Echo Left| F[Delay + Noise + Doppler]
    E -->|Echo Right| G[Delay + Noise + Doppler]
    end

    subgraph RECEIVER ["Phase 3: The Ears"]
    F --> H[Matched Filter]
    G --> I[Matched Filter]
    H --> J(LIF Encoder Left)
    I --> K(LIF Encoder Right)
    end

    subgraph BRAIN ["Phase 4: The SNN Classifier"]
    B -.->|Reference Copy| L[SNN Hidden Layer]
    J -->|Delayed Spikes| L
    K -->|Delayed Spikes| L
    L --> M[Output Layer]
    M --> N[Classification: 'Target at 5m, 30° Right']
    end

    style B fill:#f9f,stroke:#333,stroke-width:2px
    style J fill:#f9f,stroke:#333,stroke-width:2px
    style K fill:#f9f,stroke:#333,stroke-width:2px
    style L fill:#f9f,stroke:#333,stroke-width:4px
```